# PPE Detection — retrain on a **source-grouped** split

The published PPE numbers were measured on the Roboflow export's own split, and that split
leaks: **77.95%** of its test images have a near-duplicate in train/valid (ncc 0.90), and
**98.57%** of test images share a Roboflow source stem with train/valid. Its 41,730 images
come from ~5,800 source photographs — the split was made per *image* after augmentation,
not per *photograph*. On the pixel-deduplicated subset the detector scored below all five
size-matched control subsets, so the published numbers are inflated.

This notebook retrains **the same model with the same recipe** on a split whose unit is
the source photograph, so the test split shares no photograph — and no augmented or
re-exported copy of one — with training. The result is the honest number for the paper.

Nothing here re-derives the split. `tools/splits/ppe_grouped_split.csv` in the repo *is*
the split; this notebook applies it and asserts its sha256.

### Settings to choose before running
1. **Accelerator → GPU T4 x2**  (stage 1 was trained on two GPUs: `device='0,1'`)
2. **Internet → On**  (needed for `pip install` and the Roboflow download)
3. **Add-ons → Secrets** → add `ROBOFLOW_API_KEY` and attach it to this notebook
4. Then **Save Version → Save & Run All (Commit)** and close the browser — Kaggle runs it
   offline for up to 12 h and keeps the output.

### What you get
`/kaggle/working/out.zip`, containing `best_grouped.pt`, each stage's `results.csv` and
`args.yaml`, every evaluation JSON/MD, and `split_summary.json`.

> **Two stages, exactly as the original.** Stage 1 is the from-scratch run recorded in
> `best.pt` (50 epochs requested, early stopping fired at 31, batch 96, `optimizer='auto'`,
> `lr0=0.01`). Stage 2 is the refinement recorded in `best_refined.pt` (20 epochs, batch 48,
> `optimizer='SGD'`, `lr0=0.001`). Both argument sets are embedded below verbatim; only
> `data`, `project` and `name` are changed, plus stage 2's starting weights, which by
> definition point at stage 1's output rather than the original Kaggle dataset path.


## 1. GPU + CPU, and the clock

Kaggle stops a committed run at 12 h. `T_START` is used later to skip stage 2 if stage 1 ran long.


In [ ]:
import os, time, datetime, torch

T_START = time.time()
print('start (UTC):', datetime.datetime.utcfromtimestamp(T_START).isoformat(timespec='seconds'))
print('CPUs:', os.cpu_count(), '| CUDA:', torch.cuda.is_available(),
      '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    print('NO GPU -- Settings > Accelerator > GPU T4 x2, then restart.')


## 2. Install, with ultralytics pinned

Pinned to **8.4.75**, the version recorded inside `best_refined.pt`. A different
ultralytics version changes augmentation defaults and the loss, which would make the
retrained number incomparable with the published one. No `-U`: upgrading torch can pull a
build that drops support for Kaggle's GPU.


In [ ]:
!pip install -q "ultralytics==8.4.75" roboflow
import torch, ultralytics, roboflow
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__,
      '| roboflow', roboflow.__version__)
assert ultralytics.__version__ == '8.4.75', (
    f'expected ultralytics 8.4.75, got ' + ultralytics.__version__ +
    ' -- the retrain would not be comparable with the published run')


## 3. Roboflow API key

From **Add-ons → Secrets** (`ROBOFLOW_API_KEY`), same as the existing notebooks. The key is never printed.


In [ ]:
import os
ROBOFLOW_API_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
except Exception:
    pass
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
assert ROBOFLOW_API_KEY, (
    'No Roboflow API key found. Add a Kaggle Secret named ROBOFLOW_API_KEY '
    '(Add-ons -> Secrets, then attach it to this notebook), or set the '
    'ROBOFLOW_API_KEY environment variable. Get your key at '
    'https://app.roboflow.com (Settings -> API).'
)
print('Roboflow key loaded:', bool(ROBOFLOW_API_KEY))


## 4. Download the dataset (~2.8 GB)

The same export the paper used: `segp-fcn6m/ppe-yezzu-fwbjo` version 1, `yolov8` format. The split inside it is the leaky one and is about to be replaced.


In [ ]:
import shutil, glob
DEST = '/kaggle/working/ppe'
shutil.rmtree(DEST, ignore_errors=True)   # roboflow skips the download if the dir exists

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('segp-fcn6m').project('ppe-yezzu-fwbjo')
dataset = project.version(1).download('yolov8', location=DEST)

ORIG = os.path.dirname(glob.glob('/kaggle/working/ppe/**/data.yaml', recursive=True)[0])
print('dataset at:', ORIG, '|', sorted(os.listdir(ORIG)))
n = sum(len(os.listdir(os.path.join(ORIG, s, 'images'))) for s in ('train', 'valid', 'test'))
print('images:', n)
assert n == 41730, f'expected 41730 images, found {n} -- the export has changed'


## 5. Clone the repo (branch `paper-prep`)

The split definition and the tooling live in the repo, so the run is reproducible from a
commit rather than from cells pasted into Kaggle.


In [ ]:
!rm -rf /kaggle/working/repo
!git clone -q --branch paper-prep https://github.com/VanVan120/AR-PPE-Detection /kaggle/working/repo
REPO = '/kaggle/working/repo'
import subprocess
COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
print('branch  : paper-prep')
print('commit  :', COMMIT)
print(subprocess.run(['git', '-C', REPO, 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout.strip())
CSV = os.path.join(REPO, 'tools', 'splits', 'ppe_grouped_split.csv')
assert os.path.isfile(CSV), CSV
print('split csv:', CSV, '|', sum(1 for _ in open(CSV)) - 1, 'rows')


## 6. Apply the grouped split

`group_split.py --apply` re-derives nothing: it reads the CSV, hard-links every image and
label into `train/valid/test`, and writes `data.yaml`, `split_summary.json` and
`test_one_per_source.txt`. It fails loudly if any file in the CSV is missing from the
download, or if any downloaded image is not in the CSV.

The sha256 below was computed when the CSV was built locally. If it does not match, the
CSV and the dataset are not the pair this notebook was written for — stop rather than
train on a split nobody has audited.


In [ ]:
EXPECTED_SHA256 = '60d0437ee36fa6f62a2f235fd2159becbfa812309bed394cdca1b6f2b9293596'
SEED_NOTE = '43 (seed 42 rejected: worst test class-share difference 5.10 pp > 3.0 pp tolerance)'

GROUPED = '/kaggle/working/ppe_grouped'
!rm -rf {GROUPED}
!python {REPO}/tools/group_split.py --apply {CSV} --dataset-dir {ORIG} --out {GROUPED} --seed-note "43 (seed 42 rejected: worst test class-share difference 5.10 pp > 3.0 pp tolerance)"

import json
summary = json.load(open(os.path.join(GROUPED, 'split_summary.json')))
print(json.dumps({k: v['images'] for k, v in summary['splits'].items()}, indent=2))
print('sha256 :', summary['assignment_sha256'])
assert summary['assignment_sha256'] == EXPECTED_SHA256, (
    'split sha256 mismatch!\n  expected ' + EXPECTED_SHA256 +
    '\n  got      ' + summary['assignment_sha256'] +
    '\nThe CSV does not describe this dataset. Do not train.')
print('\nsplit verified against the locally built definition.')
GROUPED_YAML = os.path.join(GROUPED, 'data.yaml')
for s in ('train', 'valid', 'test'):
    d = summary['splits'][s]
    print(f"  {s:<6} {d['images']:>6} images  {d['source_groups']:>6} source groups  "
          f"{d['instances_total']:>7} boxes")


## 7. Stage 1 — the from-scratch run

Exactly the arguments recorded in the original `best.pt`, with only `data`, `project` and
`name` changed. Note `epochs=50` with `patience=12`: the original stopped early at epoch
31, and this run is free to stop wherever it stops. `optimizer='auto'` and `lr0=0.01` are
as recorded.


In [ ]:
from ultralytics import YOLO

STAGE1_ARGS = {
    'agnostic_nms': False,
    'amp': True,
    'angle': 1.0,
    'augment': False,
    'auto_augment': 'randaugment',
    'batch': 96,
    'bgr': 0.0,
    'box': 7.5,
    'cache': False,
    'cfg': None,
    'classes': None,
    'close_mosaic': 10,
    'cls': 0.5,
    'cls_pw': 0.0,
    'compile': False,
    'conf': None,
    'copy_paste': 0.0,
    'copy_paste_mode': 'flip',
    'cos_lr': False,
    'cutmix': 0.0,
    'data': '/kaggle/working/ppe/data.yaml',
    'degrees': 0.0,
    'deterministic': True,
    'device': '0,1',
    'dfl': 1.5,
    'dnn': False,
    'dropout': 0.0,
    'dynamic': False,
    'embed': None,
    'end2end': None,
    'epochs': 50,
    'erasing': 0.4,
    'exist_ok': True,
    'fliplr': 0.5,
    'flipud': 0.0,
    'format': 'torchscript',
    'fraction': 1.0,
    'freeze': None,
    'half': False,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'imgsz': 640,
    'int8': False,
    'iou': 0.7,
    'keras': False,
    'kobj': 1.0,
    'line_width': None,
    'lr0': 0.01,
    'lrf': 0.01,
    'mask_ratio': 4,
    'max_det': 300,
    'mixup': 0.0,
    'mode': 'train',
    'momentum': 0.937,
    'mosaic': 1.0,
    'multi_scale': 0.0,
    'name': 'ppe_s',
    'nbs': 64,
    'nms': False,
    'opset': None,
    'optimize': False,
    'optimizer': 'auto',
    'overlap_mask': True,
    'patience': 12,
    'perspective': 0.0,
    'plots': True,
    'pose': 12.0,
    'pretrained': True,
    'profile': False,
    'project': '/kaggle/working/runs',
    'rect': False,
    'resume': False,
    'retina_masks': False,
    'rle': 1.0,
    'save': True,
    'save_conf': False,
    'save_crop': False,
    'save_frames': False,
    'save_json': False,
    'save_period': -1,
    'save_txt': False,
    'scale': 0.5,
    'seed': 0,
    'shear': 0.0,
    'show': False,
    'show_boxes': True,
    'show_conf': True,
    'show_labels': True,
    'simplify': True,
    'single_cls': False,
    'source': None,
    'split': 'val',
    'stream_buffer': False,
    'task': 'detect',
    'time': None,
    'tracker': 'botsort.yaml',
    'translate': 0.1,
    'val': True,
    'verbose': True,
    'vid_stride': 1,
    'visualize': False,
    'warmup_bias_lr': 0.0,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'weight_decay': 0.0005,
    'workers': 8,
    'workspace': None,
}

PROJECT = '/kaggle/working/runs'
STAGE1_ARGS.update(data=GROUPED_YAML, project=PROJECT, name='grouped_s1')

print('stage 1:', {k: STAGE1_ARGS[k] for k in
      ('epochs', 'imgsz', 'batch', 'optimizer', 'lr0', 'patience', 'seed', 'device')})
m1 = YOLO('yolov8s.pt')
m1.train(**STAGE1_ARGS)

S1_DIR = os.path.join(PROJECT, 'grouped_s1')
S1_BEST = os.path.join(S1_DIR, 'weights', 'best.pt')
assert os.path.isfile(S1_BEST), S1_BEST
print('stage 1 best:', S1_BEST)
print('elapsed so far: %.2f h' % ((time.time() - T_START) / 3600))


## 8. Evaluate stage 1, and bank everything

Copied to `/kaggle/working/out/` now, so that if stage 2 runs out of time the stage-1
result still survives in the notebook output.


In [ ]:
OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)

!python {REPO}/tools/eval_grouped.py --weights {S1_BEST} --dataset-dir {GROUPED} --out-dir {OUT}/eval_stage1 --name eval_stage1

import shutil
shutil.copy(os.path.join(GROUPED, 'split_summary.json'), OUT)
os.makedirs(os.path.join(OUT, 'stage1'), exist_ok=True)
for f in ('results.csv', 'args.yaml'):
    p = os.path.join(S1_DIR, f)
    if os.path.isfile(p):
        shutil.copy(p, os.path.join(OUT, 'stage1', f))
shutil.copy(S1_BEST, os.path.join(OUT, 'stage1', 'best.pt'))
print(sorted(os.listdir(OUT)))
print(open(os.path.join(OUT, 'eval_stage1', 'eval_stage1.md')).read()[:1500])


## 9. Stage 2 — the refinement

Exactly the arguments recorded in `best_refined.pt`, with `data`, `project` and `name`
changed, and starting from **stage 1's** best checkpoint instead of the original Kaggle
dataset path — which is what "continue from stage 1" means.

Skipped if more than 8.5 h have already gone, leaving comfortable room inside Kaggle's
12 h limit for stage 2's own evaluation and the zip.


In [ ]:
STAGE2_ARGS = {
    'agnostic_nms': False,
    'amp': True,
    'angle': 1.0,
    'augment': False,
    'auto_augment': 'randaugment',
    'batch': 48,
    'bgr': 0.0,
    'box': 7.5,
    'cache': False,
    'cfg': None,
    'classes': None,
    'close_mosaic': 10,
    'cls': 0.5,
    'cls_pw': 0.0,
    'compile': False,
    'conf': None,
    'copy_paste': 0.0,
    'copy_paste_mode': 'flip',
    'cos_lr': False,
    'cutmix': 0.0,
    'data': '/kaggle/working/ppe/data.yaml',
    'degrees': 0.0,
    'deterministic': True,
    'device': '0',
    'dfl': 1.5,
    'dnn': False,
    'dropout': 0.0,
    'dynamic': False,
    'embed': None,
    'end2end': None,
    'epochs': 20,
    'erasing': 0.4,
    'exist_ok': True,
    'fliplr': 0.5,
    'flipud': 0.0,
    'format': 'torchscript',
    'fraction': 1.0,
    'freeze': None,
    'half': False,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'imgsz': 640,
    'int8': False,
    'iou': 0.7,
    'keras': False,
    'kobj': 1.0,
    'line_width': None,
    'lr0': 0.001,
    'lrf': 0.01,
    'mask_ratio': 4,
    'max_det': 300,
    'mixup': 0.0,
    'mode': 'train',
    'momentum': 0.937,
    'mosaic': 1.0,
    'multi_scale': 0.0,
    'name': 'ppe_s_cont',
    'nbs': 64,
    'nms': False,
    'opset': None,
    'optimize': False,
    'optimizer': 'SGD',
    'overlap_mask': True,
    'patience': 10,
    'perspective': 0.0,
    'plots': True,
    'pose': 12.0,
    'pretrained': True,
    'profile': False,
    'project': '/kaggle/working/runs',
    'rect': False,
    'resume': False,
    'retina_masks': False,
    'rle': 1.0,
    'save': True,
    'save_conf': False,
    'save_crop': False,
    'save_frames': False,
    'save_json': False,
    'save_period': -1,
    'save_txt': False,
    'scale': 0.5,
    'seed': 0,
    'shear': 0.0,
    'show': False,
    'show_boxes': True,
    'show_conf': True,
    'show_labels': True,
    'simplify': True,
    'single_cls': False,
    'source': None,
    'split': 'val',
    'stream_buffer': False,
    'task': 'detect',
    'time': None,
    'tracker': 'botsort.yaml',
    'translate': 0.1,
    'val': True,
    'verbose': True,
    'vid_stride': 1,
    'visualize': False,
    'warmup_bias_lr': 0.1,
    'warmup_epochs': 1.0,
    'warmup_momentum': 0.8,
    'weight_decay': 0.0005,
    'workers': 8,
    'workspace': None,
}

STAGE2_ARGS.update(data=GROUPED_YAML, project=PROJECT, name='grouped_s2')

BUDGET_H = 8.5
elapsed_h = (time.time() - T_START) / 3600
S2_BEST = None
if elapsed_h > BUDGET_H:
    print(f'SKIPPING stage 2: {elapsed_h:.2f} h already elapsed, over the {BUDGET_H} h budget.')
    print('Kaggle stops a committed run at 12 h, and stage 2 plus its evaluation would not')
    print('finish. The stage-1 weights and evaluation are already in /kaggle/working/out.')
    print('To get stage 2: re-run this notebook with stage 1 replaced by the banked')
    print('stage1/best.pt attached as a Kaggle dataset input.')
else:
    print(f'{elapsed_h:.2f} h elapsed, under the {BUDGET_H} h budget -- running stage 2.')
    print('stage 2:', {k: STAGE2_ARGS[k] for k in
          ('epochs', 'imgsz', 'batch', 'optimizer', 'lr0', 'patience', 'seed', 'device')})
    m2 = YOLO(S1_BEST)
    m2.train(**STAGE2_ARGS)
    S2_DIR = os.path.join(PROJECT, 'grouped_s2')
    S2_BEST = os.path.join(S2_DIR, 'weights', 'best.pt')
    assert os.path.isfile(S2_BEST), S2_BEST
    print('stage 2 best:', S2_BEST)
print('elapsed: %.2f h' % ((time.time() - T_START) / 3600))


## 10. Evaluate stage 2

Same four runs as stage 1, so the two stages are directly comparable.


In [ ]:
if S2_BEST:
    !python {REPO}/tools/eval_grouped.py --weights {S2_BEST} --dataset-dir {GROUPED} --out-dir {OUT}/eval_stage2 --name eval_stage2
    print(open(os.path.join(OUT, 'eval_stage2', 'eval_stage2.md')).read()[:1500])
else:
    print('stage 2 was skipped, so there is nothing to evaluate here.')


## 11. Collect the output and zip it

`/kaggle/working/out.zip` is what to download from the **Output** tab. `best_grouped.pt` is
stage 2's weights if stage 2 ran, otherwise stage 1's — the file name always means "the
model trained on the grouped split", and `PROVENANCE.txt` says which stage it came from.


In [ ]:
import shutil, json, subprocess

FINAL_DIR, FINAL_STAGE = (None, None)
if S2_BEST:
    FINAL_DIR, FINAL_STAGE = os.path.join(PROJECT, 'grouped_s2'), 'stage2'
else:
    FINAL_DIR, FINAL_STAGE = S1_DIR, 'stage1'

if FINAL_STAGE == 'stage2':
    os.makedirs(os.path.join(OUT, 'stage2'), exist_ok=True)
    for f in ('results.csv', 'args.yaml'):
        p = os.path.join(FINAL_DIR, f)
        if os.path.isfile(p):
            shutil.copy(p, os.path.join(OUT, 'stage2', f))

shutil.copy(os.path.join(FINAL_DIR, 'weights', 'best.pt'),
            os.path.join(OUT, 'best_grouped.pt'))

with open(os.path.join(OUT, 'PROVENANCE.txt'), 'w') as fh:
    fh.write('best_grouped.pt is the %s checkpoint\n' % FINAL_STAGE)
    fh.write('repo commit     : %s\n' % COMMIT)
    fh.write('split sha256    : %s\n' % EXPECTED_SHA256)
    fh.write('ultralytics     : %s\n' % ultralytics.__version__)
    fh.write('torch           : %s\n' % torch.__version__)
    fh.write('total hours     : %.2f\n' % ((time.time() - T_START) / 3600))

shutil.make_archive('/kaggle/working/out', 'zip', OUT)
print('wrote /kaggle/working/out.zip')
for r, _d, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(r, f)
        print(f'  {os.path.relpath(p, OUT):<48} {os.path.getsize(p)/1e6:8.2f} MB')
print()
print(open(os.path.join(OUT, 'PROVENANCE.txt')).read())
